# 第 7 周练习：用 QLoRA 微调 `meta-llama/Llama-3.2-3B`（价格预测）

## 练习目标（理念）

在 **Google Colab** 上走通一条完整的 **QLoRA 微调流水线**：

1. 登录 HuggingFace / W&B
2. 按 **LITE / MINIMAL** 开关选数据规模与超参
3. 4-bit 量化加载 Llama 3.2 3B
4. 用 `SFTTrainer` + LoRA 训练，并 **push 到 Hub**
5. 写 `model_predict`，调用课程提供的 `evaluate` / `Tester` 做评估

## 和第 7 周概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| QLoRA = 量化 + LoRA | `BitsAndBytesConfig` + `LoraConfig` + `SFTTrainer` |
| 提示/补全数据 | `prompt` + `completion` → 字段 `text` |
| 训练可观测性 | Weights & Biases（`wandb`） |
| 推理与评估 | `generate` + 价格解析 + `evaluate(...)` |

## 怎么跑

1. Colab GPU 运行时；Secrets 里配置 `HF_TOKEN`、`WANDB_API_KEY`
2. 需要 HuggingFace 对 Llama 的访问权限
3. `LITE_MODE=True` 用轻量数据先跑通；确认流程后再考虑全量


In [ ]:
# ========== 安装依赖，并下载课程 util.py（evaluate / Tester）==========
# 钉版本安装 bitsandbytes 与 trl（包名与版本约束保持原样）
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1
# 再装训练/推理常用栈（未钉版本的包名保持原样）
!pip install -q torch transformers accelerate peft datasets wandb matplotlib
# 从课程仓库拉取 week7/util.py，供后面 evaluate / Tester 使用
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py


In [ ]:
# ========== 导入 + HF/W&B 登录 + 固定随机种子 ==========

# 标准库 os：写 WANDB 相关环境变量
import os
# 标准库 re：从模型输出里抽价格数字
import re
# datetime：生成带时间戳的 run 名称
from datetime import datetime
# tqdm：进度条（util / 训练侧也可能用到）
from tqdm import tqdm
# Colab secrets：读 HF_TOKEN / WANDB_API_KEY，避免写进源码
from google.colab import userdata
# HuggingFace Hub 登录
from huggingface_hub import login
# PyTorch
import torch
# 模型、分词器、种子、量化配置、生成配置
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed, BitsAndBytesConfig,GenerationConfig
# 数据集加载
from datasets import load_dataset
# LoRA 配置与 PEFT 模型类型（本格主要用 LoraConfig）
from peft import LoraConfig, PeftModel
# TRL 监督微调：Trainer + 训练参数
from trl import SFTTrainer, SFTConfig
# Weights & Biases：实验追踪
import wandb
# 课程工具：evaluate、Tester（来自刚下载的 util.py）
from util import evaluate, Tester

# 从 Colab userdata 取 HF token 并登录
hf_token = userdata.get("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

# 把 W&B API key 写入环境变量后登录
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
wandb.login()

# 固定全局随机种子，便于复现
set_seed(42)


In [ ]:
# ========== 超参与运行名：LITE_MODE / MINIMAL_MODE 两套开关 ==========

# 基座模型 id（须与 Hub 上一致，勿改译）
BASE_MODEL = "meta-llama/Llama-3.2-3B"
# W&B / 本地输出用的项目名片段
PROJECT_NAME = "price"
# 推送到 Hub 时的用户名/命名空间
HF_USER = "Xander-K"
# True：用轻量数据集与更小 batch 等，先跑通流水线
LITE_MODE = True
# 数据提供者命名空间
DATA_USER = "ed-donner"
# 按 LITE_MODE 选择 lite 或 full 数据集 id
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

# True：再砍训练/验证条数，做最小冒烟
MINIMAL_MODE = False
if MINIMAL_MODE:
    # 最小模式：更少样本、更小保存上限等
    N_TRAIN = 500
    N_VAL = 100
    EVAL_SIZE = 150
    EPOCHS = 1
    BATCH_SIZE = 8
    GRADIENT_ACCUMULATION_STEPS = 2
    MAX_SEQUENCE_LENGTH = 128
    LORA_R = 32
    VAL_SIZE = 100
    LOG_STEPS = 5
    SAVE_STEPS = 50
    SAVE_TOTAL_LIMIT = 2
else:
    # 非最小：N_TRAIN/N_VAL 为 None 表示不裁剪；其它随 LITE_MODE 分支
    N_TRAIN = None
    N_VAL = None
    EVAL_SIZE = 200
    EPOCHS = 1 if LITE_MODE else 3
    BATCH_SIZE = 32 if LITE_MODE else 256
    GRADIENT_ACCUMULATION_STEPS = 1
    MAX_SEQUENCE_LENGTH = 128
    LORA_R = 32 if LITE_MODE else 256
    VAL_SIZE = 500 if LITE_MODE else 1000
    LOG_STEPS = 5 if LITE_MODE else 10
    SAVE_STEPS = 100 if LITE_MODE else 200
    SAVE_TOTAL_LIMIT = 10

# 本次 run 的时间戳名字
RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
# 后缀标记，方便在 Hub / W&B 区分
if LITE_MODE:
    RUN_NAME += "-lite"
if MINIMAL_MODE:
    RUN_NAME += "-minimal"
# 本地 output_dir / 组合名
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
# Hub 上的完整模型 id：用户名/项目-run
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# LoRA alpha 常取 2*r
LORA_ALPHA = LORA_R * 2
# 对注意力投影做 LoRA 的模块名列表
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]
LORA_DROPOUT = 0.1
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = "cosine"
WEIGHT_DECAY = 0.001
# bitsandbytes 分页 AdamW（显存友好）
OPTIMIZER = "paged_adamw_32bit"

# 读 GPU compute capability；>=8 通常可用 bf16
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

# 是否把日志打到 W&B
LOG_TO_WANDB = True
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"
if LOG_TO_WANDB:
    # 初始化一次 W&B run
    wandb.init(project=PROJECT_NAME, name=RUN_NAME)

# 打印关键开关与输出名，便于核对
print(f"MINIMAL_MODE={MINIMAL_MODE}, N_TRAIN={N_TRAIN}, N_VAL={N_VAL}")
print(f"BASE_MODEL={BASE_MODEL}, OUTPUT={PROJECT_RUN_NAME}")


In [ ]:
# ========== 加载数据集：拼 text = prompt + completion，可选裁剪 ==========

# 从 Hub 加载 DatasetDict（含 train/val/test）
dataset = load_dataset(DATASET_NAME)
train = dataset["train"]
# 验证集只取前 VAL_SIZE 条，控制评估开销
val = dataset["val"].select(range(VAL_SIZE))
test = dataset["test"]

# 最小模式：再裁剪 train / val 条数
if MINIMAL_MODE and N_TRAIN:
    train = train.select(range(min(N_TRAIN, len(train))))
if MINIMAL_MODE and N_VAL:
    val = val.select(range(min(N_VAL, len(val))))

def add_text(row):
    # SFT 需要单一 text 字段：提示 + 补全直接拼接
    return {"text": row["prompt"] + row["completion"]}

# map 写出新列 text
train = train.map(add_text)
val = val.map(add_text)
# 打印三份划分的规模
print(f"Train {len(train)}, val {len(val)}, test {len(test)}")


In [ ]:
# ========== 4-bit 量化配置 + 加载分词器与基座模型 ==========

if use_bf16:
    # Ampere+：计算 dtype 用 bfloat16
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
else:
    # 较老 GPU：用 float16 做 4-bit 计算
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )

# 分词器；trust_remote_code 与原参数一致
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
# Llama 无独立 pad：用 eos 顶上
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 量化加载基座，device_map 自动分配设备
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
# 生成配置 pad_token_id 对齐，减少警告
base_model.generation_config.pad_token_id = tokenizer.pad_token_id
# 打印显存脚印（MB）
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")


In [ ]:
# ========== LoRA 配置 + SFTConfig + 组装 SFTTrainer ==========

# LoRA：秩 r、alpha、dropout、目标模块、因果 LM 任务
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

# 训练参数：输出目录、epoch、batch、优化器、保存/日志、精度、Hub 推送等
train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    # 与 use_bf16 互斥：老卡 fp16，新卡 bf16
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    # 告诉 SFT 读哪一列当文本
    dataset_text_field="text",
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
)

# 组装 Trainer：基座 + train/val + LoRA + 参数
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters,
)
print("Trainer ready.")


In [ ]:
# ========== 开始训练，推送到 Hub，结束 W&B run ==========

# 启动监督微调（耗时与费用取决于数据/GPU）
fine_tuning.train()
# 再把适配器/模型推到 Hub（private=True 与原逻辑一致）
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to hub: {PROJECT_RUN_NAME}")
# 若开了 W&B，正式收尾本次 run
if LOG_TO_WANDB:
    wandb.finish()


In [ ]:
# ========== 推理辅助：从输出抽价格 + model_predict（generate）==========

# 训练结束后的模型句柄；切到 eval 模式
fine_tuned = fine_tuning.model
fine_tuned.eval()

def extract_price_from_output(text):
    # 优先按约定前缀 "Price is $" 右侧解析
    if "Price is $" in text:
        part = text.split("Price is $")[-1].strip().replace(",", "")
        m = re.search(r"[-+]?\d*\.?\d+", part)
        return float(m.group()) if m else 0.0
    # 否则退回：整段文本里第一个数字
    m = re.search(r"[-+]?\d*\.?\d+", text)
    return float(m.group()) if m else 0.0

def model_predict(datapoint):
    # 评估接口约定：吃一条含 prompt 的样本
    prompt = datapoint["prompt"]
    # 分词并放到模型所在设备
    inputs = tokenizer(prompt, return_tensors="pt").to(fine_tuned.device)
    with torch.no_grad():
        # 贪心生成最多 12 个新 token
        out = fine_tuned.generate(
            **inputs,
            max_new_tokens=12,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    # 解码整段（含提示），再抽价格
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    price = extract_price_from_output(decoded)
    # 返回两位小数字符串（与 evaluate 期望格式对齐）
    return f"{price:.2f}"


In [ ]:
# ========== 另一版推理：GenerationConfig + CUDA autocast（bf16）==========

# 再次取模型并 eval（本格可独立重跑推理路径）
fine_tuned = fine_tuning.model
fine_tuned.eval()

# 合法生成参数集中到 GenerationConfig，避免 temperature/top_p 等无效警告
gen_config = GenerationConfig(max_new_tokens=12, do_sample=False, pad_token_id=tokenizer.eos_token_id)

def extract_price_from_output(text):
    # 与上一格相同的价格解析逻辑
    if "Price is $" in text:
        part = text.split("Price is $")[-1].strip().replace(",", "")
        m = re.search(r"[-+]?\d*\.?\d+", part)
        return float(m.group()) if m else 0.0
    m = re.search(r"[-+]?\d*\.?\d+", text)
    return float(m.group()) if m else 0.0

def model_predict(datapoint):
    prompt = datapoint["prompt"]
    inputs = tokenizer(prompt, return_tensors="pt").to(fine_tuned.device)
    with torch.no_grad():
        # 看参数实际设备：CUDA 上用 autocast(bfloat16) 包一层 generate
        device = next(fine_tuned.parameters()).device
        if device.type == "cuda":
            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                out = fine_tuned.generate(**inputs, generation_config=gen_config)
        else:
            out = fine_tuned.generate(**inputs, generation_config=gen_config)
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    price = extract_price_from_output(decoded)
    return f"{price:.2f}"


In [ ]:
# ========== 调用课程 evaluate：在 test 上评估 model_predict ==========
# size=EVAL_SIZE 控制评估条数；预测函数与上一格定义的 model_predict 对齐
evaluate(model_predict, test, size=EVAL_SIZE)
